In [1]:
import re
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

In [2]:
# --------- Utils ---------
def extract_gsm8k_answer(text: str):
    return text.split('####')[-1].strip()

def extract_math_answer(text: str):
    """
    Extract answer for Hendrycks MATH.
    Priority: \\boxed{...}
    """
    m = re.findall(r"\\boxed\{([^}]*)\}", text)
    if len(m) > 0:
        return m[-1].strip()

    # fallback: số cuối cùng
    nums = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return nums[-1] if nums else None



### base

In [3]:
# --------- Load model ---------
model_name = "/media/volume/tucnv/LLM_Distillation/CSD/results/qwen2/nnm_0.5B_1.5B_math/3062"  # đổi model tại đây

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True
)

model.eval()


# --------- Load GSM8K ---------
dataset = load_dataset("gsm8k", "main", split="test")


# ---------- Eval (batch) ----------
batch_size = 64
correct = 0
total = len(dataset)

set_seed(42)

progress = tqdm(
    range(0, total, batch_size),
    desc="Evaluating",
    unit="batch"
)

gt_list = []
pred_list = []

for i in progress:
    batch = dataset[i:i + batch_size]

    prompts = []
    gt_answers = []

    for question, answer in zip(batch['question'], batch['answer']):
        messages = [
            # {"role": "system", "content": "Put your final answer within \\boxed{}."},
            {"role": "user", "content": question},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)
        gt_answers.append(extract_gsm8k_answer(answer))

    gt_list.extend(gt_answers)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=784,
            do_sample=False,   # greedy => pass@1
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pred, gt in zip(decoded, gt_answers):
        pred_ans = extract_math_answer(pred)
        if pred_ans is not None and gt is not None and pred_ans == gt:
            correct += 1

        pred_list.append(pred_ans)

    if i % (batch_size * 5) == 0:
        print(f"[{i}/{total}] Acc so far: {correct / (i + len(batch)):.4f}")

# --------- Result ---------
pass_at_1 = correct / total
print(f"\nPASS@1 on GSM8K: {pass_at_1:.4%}")


The tokenizer you are loading from '/media/volume/tucnv/LLM_Distillation/CSD/results/qwen2/nnm_0.5B_1.5B_math/3062' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


Evaluating:   5%|▍         | 1/21 [00:47<15:52, 47.60s/batch]

[0/1319] Acc so far: 0.0000


Evaluating:  29%|██▊       | 6/21 [03:57<09:10, 36.72s/batch]

[320/1319] Acc so far: 0.0466


...

[640/1319] Acc so far: 0.0421


Evaluating:  76%|███████▌  | 16/21 [09:40<02:50, 34.03s/batch]

[960/1319] Acc so far: 0.0416


Evaluating: 100%|██████████| 21/21 [12:30<00:00, 35.76s/batch]

[1280/1319] Acc so far: 0.0421

PASS@1 on GSM8K: 4.0940%


In [4]:
import re
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM


model.eval()


# ---------- Load Hendrycks MATH ----------
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
# dataset = dataset.select(range(200))  # debug nhanh


# ---------- Eval (batch) ----------
batch_size = 64
correct = 0
total = len(dataset)

set_seed(42)

progress = tqdm(
    range(0, total, batch_size),
    desc="Evaluating",
    unit="batch"
)

for i in progress:
    batch = dataset[i:i + batch_size]

    prompts = []
    gt_answers = []

    for question, answer in zip(batch['problem'], batch['solution']):
        messages = [
            {"role": "system", "content": "Put your final answer within \\boxed{}."},
            {"role": "user", "content": question},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)
        gt_answers.append(extract_math_answer(answer))

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=784,
            do_sample=False,   # greedy => pass@1
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pred, gt in zip(decoded, gt_answers):
        pred_ans = extract_math_answer(pred)
        if pred_ans is not None and gt is not None and pred_ans == gt:
            correct += 1

    if i % (batch_size * 5) == 0:
        print(f"[{i}/{total}] Acc so far: {correct / (i + len(batch)):.4f}")


# ---------- Result ----------
pass_at_1 = correct / total
print(f"\nPASS@1 (Hendrycks MATH): {pass_at_1:.4%}")


Evaluating:  12%|█▎        | 1/8 [00:45<05:15, 45.02s/batch]

[0/500] Acc so far: 0.0000


Evaluating:  75%|███████▌  | 6/8 [05:05<01:41, 50.91s/batch]

[320/500] Acc so far: 0.0000


Evaluating:  88%|████████▊ | 7/8 [06:15<00:53, 53.62s/batch]


KeyboardInterrupt: 

### teacher

In [ ]:
# --------- Load model ---------
model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"  # đổi model tại đây

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True
)

model.eval()


# --------- Load GSM8K ---------
dataset = load_dataset("gsm8k", "main", split="test")


# ---------- Eval (batch) ----------
batch_size = 32
correct = 0
total = len(dataset)

set_seed(42)

progress = tqdm(
    range(0, total, batch_size),
    desc="Evaluating",
    unit="batch"
)

gt_list = []
pred_list = []

for i in progress:
    batch = dataset[i:i + batch_size]

    prompts = []
    gt_answers = []

    for question, answer in zip(batch['question'], batch['answer']):
        messages = [
            {"role": "system", "content": "Put your final answer within \\boxed{}."},
            {"role": "user", "content": question},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)
        gt_answers.append(extract_gsm8k_answer(answer))

    gt_list.extend(gt_answers)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=784,
            do_sample=False,   # greedy => pass@1
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pred, gt in zip(decoded, gt_answers):
        pred_ans = extract_math_answer(pred)
        if pred_ans is not None and gt is not None and pred_ans == gt:
            correct += 1

        pred_list.append(pred_ans)

    if i % (batch_size * 10) == 0:
        print(f"[{i}/{total}] Acc so far: {correct / (i + len(batch)):.4f}")

# --------- Result ---------
pass_at_1 = correct / total
print(f"\nPASS@1 on GSM8K: {pass_at_1:.4%}")


Evaluating:   2%|▏         | 1/42 [00:26<17:53, 26.17s/batch]

[0/1319] Acc so far: 12.5000


Evaluating:  26%|██▌       | 11/42 [03:53<10:56, 21.18s/batch]

[320/1319] Acc so far: 0.9286


Evaluating:  50%|█████     | 21/42 [07:26<07:13, 20.64s/batch]

...

[960/1319] Acc so far: 0.8607


Evaluating:  98%|█████████▊| 41/42 [14:08<00:20, 20.15s/batch]

[1280/1319] Acc so far: 0.8510


Evaluating: 100%|██████████| 42/42 [14:22<00:00, 20.52s/batch]


PASS@1 on GSM8K: 83.1691%


In [15]:
import re
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM


model.eval()


# ---------- Load Hendrycks MATH ----------
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
# dataset = dataset.select(range(200))  # debug nhanh


# ---------- Eval (batch) ----------
batch_size = 32
correct = 0
total = len(dataset)

set_seed(42)

progress = tqdm(
    range(0, total, batch_size),
    desc="Evaluating",
    unit="batch"
)

for i in progress:
    batch = dataset[i:i + batch_size]

    prompts = []
    gt_answers = []

    for question, answer in zip(batch['problem'], batch['solution']):
        messages = [
            {"role": "system", "content": "Put your final answer within \\boxed{}."},
            {"role": "user", "content": question},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)
        gt_answers.append(extract_math_answer(answer))

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=784,
            do_sample=False,   # greedy => pass@1
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pred, gt in zip(decoded, gt_answers):
        pred_ans = extract_math_answer(pred)
        if pred_ans is not None and gt is not None and pred_ans == gt:
            correct += 1

    if i % (batch_size * 10) == 0:
        print(f"[{i}/{total}] Acc so far: {correct / (i + len(batch)):.4f}")


# ---------- Result ----------
pass_at_1 = correct / total
print(f"\nPASS@1 (Hendrycks MATH): {pass_at_1:.4%}")


Evaluating:   6%|▋         | 1/16 [00:26<06:39, 26.62s/batch]

[0/500] Acc so far: 3.0000


Evaluating:  69%|██████▉   | 11/16 [04:59<02:16, 27.38s/batch]

[320/500] Acc so far: 0.6227


Evaluating: 100%|██████████| 16/16 [07:14<00:00, 27.16s/batch]


PASS@1 (Hendrycks MATH): 56.4000%


### distillm2

In [21]:
# --------- Load model ---------
model_name = "distillm-2-master/outputs/qwen2.5-math-distillm2/checkpoint-7500"  # đổi model tại đây

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="cuda:0",
    trust_remote_code=True
)

model.eval()


# --------- Load GSM8K ---------
dataset = load_dataset("gsm8k", "main", split="test")


# ---------- Eval (batch) ----------
batch_size = 32
correct = 0
total = len(dataset)

set_seed(42)

progress = tqdm(
    range(0, total, batch_size),
    desc="Evaluating",
    unit="batch"
)

gt_list = []
pred_list = []

for i in progress:
    batch = dataset[i:i + batch_size]

    prompts = []
    gt_answers = []

    for question, answer in zip(batch['question'], batch['answer']):
        messages = [
            {"role": "system", "content": "Put your final answer within \\boxed{}."},
            {"role": "user", "content": question},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)
        gt_answers.append(extract_gsm8k_answer(answer))

    gt_list.extend(gt_answers)

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=784,
            do_sample=False,   # greedy => pass@1
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pred, gt in zip(decoded, gt_answers):
        pred_ans = extract_math_answer(pred)
        if pred_ans is not None and gt is not None and pred_ans == gt:
            correct += 1

        pred_list.append(pred_ans)

    if i % (batch_size * 10) == 0:
        print(f"[{i}/{total}] Acc so far: {correct / (i + len(batch)):.4f}")

# --------- Result ---------
pass_at_1 = correct / total
print(f"\nPASS@1 on GSM8K: {pass_at_1:.4%}")


Evaluating:   2%|▏         | 1/42 [00:22<15:30, 22.70s/batch]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[0/1319] Acc so far: 4.0000


Evaluating:  26%|██▌       | 11/42 [04:10<11:48, 22.87s/batch]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[320/1319] Acc so far: 0.5124


Evaluating:  50%|█████     | 21/42 [07:58<07:59, 22.85s/batch]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

[960/1319] Acc so far: 0.5125


Evaluating:  98%|█████████▊| 41/42 [15:39<00:22, 22.92s/batch]Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[1280/1319] Acc so far: 0.5000


Evaluating: 100%|██████████| 42/42 [16:01<00:00, 22.90s/batch]


PASS@1 on GSM8K: 48.7491%


In [24]:
import re
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM


# ---------- Load model ----------
# model_name = "distillm-2-master/outputs/qwen2.5-math-distillm2/checkpoint-7500"   # đổi model nếu cần

# tokenizer = AutoTokenizer.from_pretrained(
#     model_name,
#     trust_remote_code=True
# )
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "left"

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="cuda:0",
#     torch_dtype=torch.float16,
#     trust_remote_code=True
# )

model.eval()


# ---------- Load Hendrycks MATH ----------
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test")
# dataset = dataset.select(range(200))  # debug nhanh


# ---------- Eval (batch) ----------
batch_size = 32
correct = 0
total = len(dataset)

set_seed(42)

progress = tqdm(
    range(0, total, batch_size),
    desc="Evaluating",
    unit="batch"
)

for i in progress:
    batch = dataset[i:i + batch_size]

    prompts = []
    gt_answers = []

    for question, answer in zip(batch['problem'], batch['solution']):
        messages = [
            {"role": "system", "content": "Put your final answer within \\boxed{}."},
            {"role": "user", "content": question},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        prompts.append(prompt)
        gt_answers.append(extract_math_answer(answer))

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=784,
            do_sample=False,   # greedy => pass@1
            pad_token_id=tokenizer.eos_token_id
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for pred, gt in zip(decoded, gt_answers):
        pred_ans = extract_math_answer(pred)
        if pred_ans is not None and gt is not None and pred_ans == gt:
            correct += 1

    if i % (batch_size * 10) == 0:
        print(f"[{i}/{total}] Acc so far: {correct / (i + len(batch)):.4f}")


# ---------- Result ----------
pass_at_1 = correct / total
print(f"\nPASS@1 (Hendrycks MATH): {pass_at_1:.4%}")


Evaluating:   6%|▋         | 1/16 [00:23<05:45, 23.02s/batch]

[0/500] Acc so far: 0.3333


Evaluating:  69%|██████▉   | 11/16 [04:24<02:06, 25.21s/batch]

[320/500] Acc so far: 0.2178


Evaluating: 100%|██████████| 16/16 [06:19<00:00, 23.71s/batch]


PASS@1 (Hendrycks MATH): 21.8000%
